# Data Preparation

Unified preparation pipeline for modelling. Reads the base dataset and produces
demand-type-segmented files ready for feature engineering.

**Input:**
- `datasets/base/ds_sales_timeseries.parquet` - full weekly base dataset
- `dw_dim_time_series.parquet` - time series registry

**Output:**
- `datasets/refined/ds_sales_timeseries_{demand_type}.parquet` - one file per demand type
- `datasets/excluded/ds_sales_timeseries_excluded.parquet` - series filtered out
- `tags/ds_tags.parquet` - per-series metadata (segment, demand type, outlier count)

**Steps:**
1. Maturity segmentation - filter out cold-start and discontinued series.
2. Outlier detection and capping - rolling median + MAD, non-destructive capping.
3. Demand classification - Syntetos-Boylan (ADI x CV2).
4. Save outputs.

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# -- Paths -----------------------------------------------------------------
PATH_BASE_DATASET = '../../data/gold/forecasting/datasets/base/ds_sales_timeseries.parquet'
PATH_DIM_TS       = '../../data/gold/data_warehouse/dw_dim_time_series.parquet'

PATH_REFINED  = '../../data/gold/forecasting/datasets/refined'
PATH_EXCLUDED = '../../data/gold/forecasting/datasets/excluded/ds_sales_timeseries_excluded.parquet'
PATH_TAGS     = '../../data/gold/forecasting/tags/ds_tags.parquet'

os.makedirs(PATH_REFINED, exist_ok=True)
os.makedirs(os.path.dirname(PATH_EXCLUDED), exist_ok=True)
os.makedirs(os.path.dirname(PATH_TAGS), exist_ok=True)

df_base   = pd.read_parquet(PATH_BASE_DATASET)
df_dim_ts = pd.read_parquet(PATH_DIM_TS)

print(f'Base dataset : {len(df_base):,} rows  |  {df_base["time_series_id"].nunique():,} series')

## 1. Maturity Segmentation

In [ ]:
def segment_by_maturity(df_dim_ts, reference_date=None, discontinued_weeks=26):
    df = df_dim_ts.copy()
    df['last_week_date'] = pd.to_datetime(df['last_week_date'])

    if reference_date is None:
        reference_date = df['last_week_date'].max()
    reference_date = pd.to_datetime(reference_date)

    discontinued_threshold = reference_date - pd.Timedelta(weeks=discontinued_weeks)
    df['weeks_since_last_sale'] = ((reference_date - df['last_week_date']).dt.days / 7).astype(int)
    df['segment'] = 'unknown'

    df.loc[df['total_weeks_length'] < 26, 'segment'] = 'cold_start'
    df.loc[
        (df['segment'] != 'cold_start') & (df['last_week_date'] < discontinued_threshold),
        'segment'
    ] = 'discontinued'
    df.loc[df['segment'] == 'unknown', 'segment'] = 'valid'

    return df[['time_series_id', 'segment', 'weeks_since_last_sale']]


df_segments = segment_by_maturity(df_dim_ts)
valid_ids   = df_segments.loc[df_segments['segment'] == 'valid', 'time_series_id']
df_valid    = df_base[df_base['time_series_id'].isin(valid_ids)].copy()

print(f'Valid dataset : {df_valid.shape[0]:,} rows  |  {df_valid["time_series_id"].nunique():,} series')
print()
print(df_segments['segment'].value_counts().to_string())

## 2. Outlier Detection and Capping

In [ ]:
def detect_outliers_rolling_median(ts_data, window=13, cutoff_multiplier=3.0):
    n = len(ts_data)
    outlier_mask     = np.zeros(n, dtype=bool)
    rolling_baseline = np.full(n, np.nan)
    rolling_mad      = np.full(n, np.nan)

    if (ts_data > 0).sum() < window:
        nan = np.full(n, np.nan)
        return outlier_mask, nan, nan, nan

    for i in range(n):
        start = max(0, i - window // 2)
        end   = min(n, i + window // 2 + 1)
        w = ts_data[start:end]
        w = w[w > 0]
        if len(w) < 3:
            continue
        baseline            = np.median(w)
        rolling_baseline[i] = baseline
        rolling_mad[i]      = np.median(np.abs(w - baseline))

    rolling_std = 1.4826 * rolling_mad
    upper       = rolling_baseline + cutoff_multiplier * rolling_std
    lower       = np.zeros(n)

    for i in range(n):
        if ts_data[i] > 0 and not np.isnan(upper[i]):
            if ts_data[i] > upper[i] or ts_data[i] < lower[i]:
                outlier_mask[i] = True

    return outlier_mask, rolling_baseline, rolling_std, upper


def run_outlier_pipeline(df, window=13, cutoff_multiplier=3.0):
    results = []
    for ts_id, group in df.groupby('time_series_id'):
        group = group.sort_values('week_date').copy()
        y     = group['units_sold'].astype(float).values
        out_mask, baseline, roll_std, upper = detect_outliers_rolling_median(
            y, window=window, cutoff_multiplier=cutoff_multiplier
        )
        y_cap = y.copy()
        valid_upper = ~np.isnan(upper)
        y_cap[out_mask & valid_upper] = upper[out_mask & valid_upper]

        group['is_outlier']       = out_mask
        group['rolling_baseline'] = baseline
        group['rolling_std']      = roll_std
        group['upper_threshold']  = upper
        group['lower_threshold']  = 0.0
        group['y_capped']         = y_cap.astype(int)
        results.append(group)
    return pd.concat(results, ignore_index=True)


df_treated = run_outlier_pipeline(df_valid, window=13, cutoff_multiplier=3.0)

n_outliers = df_treated['is_outlier'].sum()
n_series   = df_treated['time_series_id'].nunique()
print(f'Outliers detected : {n_outliers:,}  ({n_outliers / len(df_treated) * 100:.2f}% of all observations)')
print(f'Series affected   : {df_treated.groupby("time_series_id")["is_outlier"].any().sum():,} / {n_series:,}')

## 3. Demand Classification (Croston)

In [ ]:
def classify_series(y: 'pd.Series') -> str:
    non_zero = y[y > 0]
    if len(non_zero) == 0:
        return 'lumpy'
    adi = len(y) / len(non_zero)
    cv2 = (non_zero.std(ddof=0) / non_zero.mean()) ** 2 if non_zero.mean() != 0 else 0
    if adi < 1.32:
        return 'smooth' if cv2 < 0.49 else 'erratic'
    return 'intermittent' if cv2 < 0.49 else 'lumpy'


demand_type_map = (
    df_treated
    .groupby('time_series_id')['y_capped']
    .apply(classify_series)
)
df_treated['demand_type'] = df_treated['time_series_id'].map(demand_type_map)

print('Demand classification:')
print(df_treated.groupby('demand_type')['time_series_id'].nunique()
      .rename('n_series').sort_values(ascending=False).to_string())

## 4. Save Outputs

In [ ]:
# 4.1 Refined datasets - standard time series format: unique_id | ds | y
DEMAND_TYPES = ['smooth', 'erratic', 'intermittent', 'lumpy']

for demand_type in DEMAND_TYPES:
    df_out = (
        df_treated[df_treated['demand_type'] == demand_type]
        [['time_series_id', 'week_date', 'y_capped', 'demand_type']]
        .rename(columns={'time_series_id': 'unique_id', 'week_date': 'ds', 'y_capped': 'y'})
        .sort_values(['unique_id', 'ds'])
        .reset_index(drop=True)
    )
    path = f'{PATH_REFINED}/ds_sales_timeseries_{demand_type}.parquet'
    df_out.to_parquet(path, index=False)
    print(f'  Saved {demand_type:<15}: {df_out["unique_id"].nunique():,} series  |  {len(df_out):,} rows')

In [ ]:
# 4.2 Excluded series
excluded_ids = df_segments[df_segments['segment'] != 'valid'][
    ['time_series_id', 'segment', 'weeks_since_last_sale']
]
df_excluded = (
    df_base[df_base['time_series_id'].isin(excluded_ids['time_series_id'])]
    .merge(excluded_ids, on='time_series_id', how='left')
)
df_excluded.to_parquet(PATH_EXCLUDED, index=False)
print(f'Excluded : {df_excluded["time_series_id"].nunique():,} series  ->  {PATH_EXCLUDED}')

In [ ]:
# 4.3 Tags - per-series metadata for all series (valid and excluded)
outlier_counts = (
    df_treated.groupby('time_series_id')['is_outlier']
    .sum()
    .rename('n_outliers')
    .reset_index()
)
df_tags = (
    df_segments
    .merge(outlier_counts, on='time_series_id', how='left')
    .merge(
        df_treated[['time_series_id', 'demand_type']].drop_duplicates(),
        on='time_series_id', how='left',
    )
)
df_tags.to_parquet(PATH_TAGS, index=False)
print(f'Tags     : {len(df_tags):,} series  ->  {PATH_TAGS}')

## Summary

In [ ]:
print('OUTPUTS GENERATED')
print('=' * 60)
print()
print('  datasets/refined/')
for dt in DEMAND_TYPES:
    n = df_treated[df_treated['demand_type'] == dt]['time_series_id'].nunique()
    print(f'    ds_sales_timeseries_{dt}.parquet   {n:>5,} series')
print()
print('  datasets/excluded/')
print(f'    ds_sales_timeseries_excluded.parquet   '
      f'{df_excluded["time_series_id"].nunique():>5,} series')
print()
print('  tags/')
print(f'    ds_tags.parquet   {len(df_tags):,} total series')